# Tanzania VACS 2024 — PUD exploration

Single public file: **`TANZANIA_VACS_2024_PUD.dta`** (`data/raw/Tanzania 2024/`). Males and females are **in one dataset** (split-sample EAs; use **`SEX`**).

**Flow:** (1) Load → (2) **§2 — column list & quick EDA** → **checklist** (`utils.checklist`, **`type_and_width`** + TSV) → (3) **§3 — further EDA** (row samples, derived HH fields, slot summaries, duplicates) → (4) **§4 — harmonized codebook**.

**Reusing for other countries:** duplicate this notebook, change `COUNTRY_*` paths and the mapping dict in the harmonized markdown if variable names differ.

In [1]:
from pathlib import Path
import sys

from IPython.display import display

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyreadstat
import re

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# --- Tanzania 2024 (edit for another country) ---
COUNTRY_DIR = ROOT / "data" / "raw" / "Tanzania 2024"
PUD_NAME = "TANZANIA_VACS_2024_PUD.dta"
DTA_PATH = COUNTRY_DIR / PUD_NAME

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")

## 1. Load data

`pyreadstat.read_dta` returns `df` and `meta` (`column_names_to_labels`, value labels when present).

In [2]:
if not DTA_PATH.is_file():
    raise FileNotFoundError(f"Expected:\n  {DTA_PATH}")

df, meta = pyreadstat.read_dta(DTA_PATH)
print(f"File: {DTA_PATH}")
print(f"Rows × columns: {df.shape[0]:,} × {df.shape[1]:,}")
if getattr(meta, "file_label", None):
    print(f"Stata dataset label: {meta.file_label!r}")
if "SEX" in df.columns:
    print("SEX counts:")
    display(df["SEX"].value_counts(dropna=False))
df.head()

File: /Users/starsrain/research_side_projects_ipv/data/raw/Tanzania 2024/TANZANIA_VACS_2024_PUD.dta
Rows × columns: 11,414 × 635
SEX counts:


SEX
Female    8441
Male      2973
Name: count, dtype: int64

,PSU,REGION,AREA,AGE,SEX,SCHHL,MARNOW,WVMHOM,SAMPLEWEIGHT,HIVWEIGHT,...,STID1A,STIFA,STIXAA,STIXBA,STIXCA,STIXDA,SHNUMA,MAROTHERF,MAROTHERM,PUD_ID
0,101.0,Dodoma,URBAN,24.0,Female,2.0,2.0,3.0,2217.420111,2067.27604,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,NaN,Female_Female_Dodoma _URBAN_101-101_1
1,101.0,Dodoma,URBAN,21.0,Female,2.0,2.0,2.0,2217.420111,2067.27604,...,2.0,2.0,2.0,2.0,2.0,2.0,1.0,2.0,NaN,Female_Female_Dodoma _URBAN_101-101_10
2,101.0,Dodoma,URBAN,18.0,Female,4.0,NaN,3.0,4434.840221,4134.55208,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,NaN,NaN,Female_Female_Dodoma _URBAN_101-101_11
3,101.0,Dodoma,URBAN,16.0,Female,NaN,NaN,3.0,2217.420111,2067.27604,...,2.0,2.0,2.0,98.0,2.0,2.0,NaN,NaN,NaN,Female_Female_Dodoma _URBAN_101-101_12
4,101.0,Dodoma,URBAN,16.0,Female,2.0,NaN,1.0,2217.420111,2067.27604,...,2.0,2.0,2.0,2.0,2.0,2.0,NaN,NaN,NaN,Female_Female_Dodoma _URBAN_101-101_13


## 2. Column list & quick EDA

Stata labels, dtypes, and missingness: full `var_table` preview (first 40 columns), the 15 columns with highest missingness, and `df.info`. Then run the **checklist** cells (**`utils.checklist`**); edit **`CANDIDATES`** if needed.


In [3]:
name_to_label = dict(meta.column_names_to_labels) if meta.column_names_to_labels else {}
var_table = pd.DataFrame({
    "column": df.columns,
    "stata_label": [name_to_label.get(c, "") or "" for c in df.columns],
    "dtype": df.dtypes.astype(str).values,
    "missing_n": df.isna().sum().values,
    "missing_pct": (100 * df.isna().mean()).round(2),
})

print(f"Variables: {len(df.columns):,}  |  Observations: {len(df):,}")
print(f"Embedded Stata value-label maps: {len(meta.value_labels or {})}")
display(var_table.head(40))
display(
    var_table.sort_values("missing_pct", ascending=False)
    .head(15)
    .reset_index(drop=True)
)
df.info(max_cols=20)

Variables: 635  |  Observations: 11,414
Embedded Stata value-label maps: 0


,column,stata_label,dtype,missing_n,missing_pct
PSU,PSU,PSU,float64,0,0.00
REGION,REGION,REGION,str,0,0.00
AREA,AREA,Area,str,0,0.00
AGE,AGE,,float64,0,0.00
SEX,SEX,,str,0,0.00
SCHHL,SCHHL,,float64,4866,42.63
MARNOW,MARNOW,,float64,8712,76.33
WVMHOM,WVMHOM,,float64,0,0.00
SAMPLEWEIGHT,SAMPLEWEIGHT,,float64,0,0.00
HIVWEIGHT,HIVWEIGHT,HIVWEIGHT,float64,2459,21.54


,column,stata_label,dtype,missing_n,missing_pct
0,H2_CC,,float64,11414,100.00
1,H3_CC,,float64,11414,100.00
2,PRESSWHO,,float64,11413,99.99
3,SH3AFD,,float64,11413,99.99
4,SH3BAD,,float64,11412,99.98
5,SH2AFD,,float64,11411,99.97
6,PRESSINTWHO,,float64,11410,99.96
7,SH2BAD,,float64,11410,99.96
8,SH1HIVPDN,,float64,11403,99.90
9,PREPDREAM,,float64,11401,99.89


<class 'pandas.DataFrame'>
RangeIndex: 11414 entries, 0 to 11413
Columns: 635 entries, PSU to PUD_ID
dtypes: float64(607), str(28)
memory usage: 55.3 MB


### Harmonized geography / ID checklist (`utils.checklist`)

**Admin 2 = enumeration areas (EAs):** **`CLUSTER`** (hyphenated **`NNN-NNN`** string in this extract) is the EA / cluster identifier for this project—not a substitute with **`PSU`** without checking the User Guide (**`PSU`** is numeric design).

**Geo:** **`REGION`** (coarsest), **`AREA`** (urban/rural-type layer). Edit **`CANDIDATES`** after §2 if you add columns.

See **`skills/memory.md`** for PI width / layout conventions.


In [8]:
# You choose candidates after §2 EDA; utils summarize columns present in `df`.
from utils.checklist import build_checklist_df, checklist_to_tsv

CANDIDATES = [
    ("Admin 1 (region)", ["REGION"]),
    ("Area (urban/rural etc.)", ["AREA"]),
    ("Admin 2 — enumeration area (EA)", ["CLUSTER"]),
    ("PSU (numeric design)", ["PSU"]),
    ("Stratum", ["STRATA"]),
    ("Respondent ID", ["PUD_ID"]),
    ("Roster / HH context", ["NTOT", "REGI_C"]),
    ("Sex", ["SEX"]),
    ("Weights", ["SAMPLEWEIGHT", "HIVWEIGHT"]),
]

_labels = meta.column_names_to_labels or {}
checklist_df = build_checklist_df(df, CANDIDATES, column_labels=_labels)

with pd.option_context("display.max_colwidth", 100, "display.width", 220):
    display(checklist_df)

print("\n--- TSV (copy for Excel / codebook) ---\n")
print(checklist_to_tsv(checklist_df))


,slot,column,stata_label,type_and_width,suggested_layout,dtype,nunique,missing_n,missing_pct,pi_digits_char_usual_display,min_nonnull,max_nonnull,sample_first_3,slot_notes
0,Admin 1 (region),REGION,REGION,str; 4–16 digits,<NA>,str,31,0,0.00,4–16,<NA>,<NA>,"'Dodoma', 'Dodoma', 'Dodoma'",<NA>
1,Area (urban/rural etc.),AREA,Area,str; 5 digits,<NA>,str,2,0,0.00,5,<NA>,<NA>,"'URBAN', 'URBAN', 'URBAN'",<NA>
2,Admin 2 — enumeration area (EA),CLUSTER,,str; 7 digits,<NA>,str,499,0,0.00,7,<NA>,<NA>,"'101-101', '101-101', '101-101'",<NA>
3,PSU (numeric design),PSU,PSU,float; 3 digits,<NA>,float64,499,0,0.00,3,101.0,931.0,"101.0, 101.0, 101.0",<NA>
4,Stratum,STRATA,,str; 29 digits,<NA>,str,120,0,0.00,29,<NA>,<NA>,"'Female_Dodoma _URBAN', 'Female_Dodoma _URBAN', 'Female_Dodoma _URBAN'",<NA>
5,Respondent ID,PUD_ID,,str; 44–47 digits,<NA>,str,11353,0,0.00,44–47,<NA>,<NA>,"'Female_Female_Dodoma _URBAN_1…, 'Female_Female_Dodoma _URBAN_1…, 'Female_Fema...",<NA>
6,Roster / HH context,NTOT,,float; 1–2 digits,<NA>,float64,29,0,0.00,1–2,1.0,30.0,"3.0, 2.0, 6.0",<NA>
7,Roster / HH context,REGI_C,,float; 1–2 digits,<NA>,float64,31,0,0.00,1–2,1.0,55.0,"1.0, 1.0, 1.0",<NA>
8,Sex,SEX,,str; 4–6 digits,<NA>,str,2,0,0.00,4–6,<NA>,<NA>,"'Female', 'Female', 'Female'",<NA>
9,Weights,SAMPLEWEIGHT,,float; 14–18 digits,<NA>,float64,1334,0,0.00,14–18,4.98312,38791.932325,"2217.4201107176523, 2217.4201107176523, 4434.840221435305",<NA>



--- TSV (copy for Excel / codebook) ---

slot	column	stata_label	type_and_width	suggested_layout	dtype	nunique	missing_n	missing_pct	pi_digits_char_usual_display	min_nonnull	max_nonnull	sample_first_3	slot_notes
Admin 1 (region)	REGION	REGION	str; 4–16 digits		str	31	0	0.0	4–16			'Dodoma', 'Dodoma', 'Dodoma'	
Area (urban/rural etc.)	AREA	Area	str; 5 digits		str	2	0	0.0	5			'URBAN', 'URBAN', 'URBAN'	
Admin 2 — enumeration area (EA)	CLUSTER		str; 7 digits		str	499	0	0.0	7			'101-101', '101-101', '101-101'	
PSU (numeric design)	PSU	PSU	float; 3 digits		float64	499	0	0.0	3	101.0	931.0	101.0, 101.0, 101.0	
Stratum	STRATA		str; 29 digits		str	120	0	0.0	29			'Female_Dodoma          _URBAN', 'Female_Dodoma          _URBAN', 'Female_Dodoma          _URBAN'	
Respondent ID	PUD_ID		str; 44–47 digits		str	11353	0	0.0	44–47			'Female_Female_Dodoma          _URBAN_1…, 'Female_Female_Dodoma          _URBAN_1…, 'Female_Female_Dodoma          _URBAN_1…	
Roster / HH context	NTOT		float; 1–2 digits		floa

## 3. Further EDA and exploration

### Raw row samples

Full width is large; below: **head / tail / sample** plus a **core** slice for IDs, geography, cluster, and weights.


In [7]:
pd.set_option("display.max_columns", 35)
pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 60)

print("--- head(6) ---")
display(df.head(6))
print("\n--- tail(3) ---")
display(df.tail(3))
print("\n--- sample(5, random_state=0) ---")
display(df.sample(5, random_state=0))

_core_cols = [
    c
    for c in [
        "PUD_ID",
        "PSU",
        "CLUSTER",
        "REGION",
        "REGI_C",
        "AREA",
        "STRATA",
        "SEX",
        "AGE",
        "NTOT",
        "SAMPLEWEIGHT",
        "HIVWEIGHT",
    ]
    if c in df.columns
]
subset = df[_core_cols]
print(f"\n--- core columns ({len(_core_cols)}) ---")
display(subset.head(8))
display(subset.sample(5, random_state=1))

--- head(6) ---


,PSU,REGION,AREA,AGE,SEX,SCHHL,MARNOW,WVMHOM,SAMPLEWEIGHT,HIVWEIGHT,STRATA,CLUSTER,REGI_C,NTOT,H1,H1_1,H2,...,SUIINTA,SUIATTA,STIAA,STIBA,STICA,STID1A,STIFA,STIXAA,STIXBA,STIXCA,STIXDA,SHNUMA,MAROTHERF,MAROTHERM,PUD_ID,HH_LINE,HH_KEY
0,101.0,Dodoma,URBAN,24.0,Female,2.0,2.0,3.0,2217.420111,2067.27604,Female_Dodoma _URBAN,101-101,1.0,3.0,11:02:00.000+03:00,1.0,1.0,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,NaN,Female_Female_Dodoma _URBAN_101-101_1,1,101-101_1
1,101.0,Dodoma,URBAN,21.0,Female,2.0,2.0,2.0,2217.420111,2067.27604,Female_Dodoma _URBAN,101-101,1.0,2.0,10:26:00.000+03:00,2.0,1.0,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,1.0,2.0,NaN,Female_Female_Dodoma _URBAN_101-101_10,10,101-101_10
2,101.0,Dodoma,URBAN,18.0,Female,4.0,NaN,3.0,4434.840221,4134.55208,Female_Dodoma _URBAN,101-101,1.0,6.0,12:50:00.000+03:00,1.0,1.0,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,NaN,NaN,Female_Female_Dodoma _URBAN_101-101_11,11,101-101_11
3,101.0,Dodoma,URBAN,16.0,Female,NaN,NaN,3.0,2217.420111,2067.27604,Female_Dodoma _URBAN,101-101,1.0,6.0,10:20:00.000+03:00,2.0,1.0,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,98.0,2.0,2.0,NaN,NaN,NaN,Female_Female_Dodoma _URBAN_101-101_12,12,101-101_12
4,101.0,Dodoma,URBAN,16.0,Female,2.0,NaN,1.0,2217.420111,2067.27604,Female_Dodoma _URBAN,101-101,1.0,7.0,10:57:00.000+03:00,1.0,1.0,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,NaN,NaN,NaN,Female_Female_Dodoma _URBAN_101-101_13,13,101-101_13
5,101.0,Dodoma,URBAN,18.0,Female,2.0,NaN,2.0,2217.420111,2067.27604,Female_Dodoma _URBAN,101-101,1.0,6.0,11:23:00.000+03:00,2.0,1.0,...,2.0,2.0,2.0,2.0,1.0,2.0,2.0,2.0,2.0,2.0,2.0,NaN,NaN,NaN,Female_Female_Dodoma _URBAN_101-101_14,14,101-101_14



--- tail(3) ---


,PSU,REGION,AREA,AGE,SEX,SCHHL,MARNOW,WVMHOM,SAMPLEWEIGHT,HIVWEIGHT,STRATA,CLUSTER,REGI_C,NTOT,H1,H1_1,H2,...,SUIINTA,SUIATTA,STIAA,STIBA,STICA,STID1A,STIFA,STIXAA,STIXBA,STIXCA,STIXDA,SHNUMA,MAROTHERF,MAROTHERM,PUD_ID,HH_LINE,HH_KEY
11411,931.0,Kusini Pemba,URBAN,13.0,Male,NaN,NaN,1.0,44.797915,NaN,Male _Kusini Pemba _URBAN,931-931,55.0,2.0,11:23:00.000+03:00,1.0,2.0,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,NaN,NaN,NaN,Male_Male _Kusini Pemba _URBAN_931-931_7,7,931-931_7
11412,931.0,Kusini Pemba,URBAN,24.0,Male,2.0,2.0,1.0,44.797915,50.570151,Male _Kusini Pemba _URBAN,931-931,55.0,3.0,08:55:00.000+03:00,1.0,1.0,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,NaN,NaN,1.0,Male_Male _Kusini Pemba _URBAN_931-931_8,8,931-931_8
11413,931.0,Kusini Pemba,URBAN,17.0,Male,NaN,NaN,1.0,44.797915,50.570151,Male _Kusini Pemba _URBAN,931-931,55.0,12.0,10:47:00.000+03:00,2.0,2.0,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,NaN,NaN,NaN,Male_Male _Kusini Pemba _URBAN_931-931_9,9,931-931_9



--- sample(5, random_state=0) ---


,PSU,REGION,AREA,AGE,SEX,SCHHL,MARNOW,WVMHOM,SAMPLEWEIGHT,HIVWEIGHT,STRATA,CLUSTER,REGI_C,NTOT,H1,H1_1,H2,...,SUIINTA,SUIATTA,STIAA,STIBA,STICA,STID1A,STIFA,STIXAA,STIXBA,STIXCA,STIXDA,SHNUMA,MAROTHERF,MAROTHERM,PUD_ID,HH_LINE,HH_KEY
11234,924.0,Kaskazini Pemba,RURAL,14.0,Male,NaN,NaN,1.0,145.273354,NaN,Male _Kaskazini Pemba _RURAL,924-924,54.0,8.0,11:47:00.000+03:00,2.0,1.0,...,99.0,99.0,99.0,98.0,2.0,1.0,1.0,1.0,1.0,1.0,2.0,NaN,NaN,NaN,Male_Male _Kaskazini Pemba _RURAL_924-924_24,24,924-924_24
5242,326.0,Shinyanga,URBAN,19.0,Female,4.0,NaN,1.0,309.204818,302.727282,Female_Shinyanga _URBAN,326-326,17.0,4.0,09:40:00.000+03:00,2.0,1.0,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,NaN,NaN,NaN,Female_Female_Shinyanga _URBAN_326-326_23,23,326-326_23
7563,434.0,Mjini Magharibi,URBAN,15.0,Female,NaN,NaN,1.0,362.163492,419.032897,Female_Mjini Magharibi _URBAN,434-434,53.0,10.0,13:35:00.000+03:00,1.0,1.0,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,NaN,NaN,NaN,Female_Female_Mjini Magharibi _URBAN_434-434_11,11,434-434_11
2817,222.0,Mbeya,RURAL,13.0,Female,NaN,NaN,1.0,164.583035,NaN,Female_Mbeya _RURAL,222-222,12.0,4.0,12:50:00.000+03:00,2.0,1.0,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,NaN,NaN,NaN,Female_Female_Mbeya _RURAL_222-222_15,15,222-222_15
6616,394.0,Manyara,RURAL,21.0,Female,2.0,2.0,1.0,7127.973124,6112.081498,Female_Manyara _RURAL,394-394,21.0,12.0,10:00:00.000+03:00,1.0,1.0,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,1.0,2.0,NaN,Female_Female_Manyara _RURAL_394-394_14,14,394-394_14



--- core columns (12) ---


,PUD_ID,PSU,CLUSTER,REGION,REGI_C,AREA,STRATA,SEX,AGE,NTOT,SAMPLEWEIGHT,HIVWEIGHT
0,Female_Female_Dodoma _URBAN_101-101_1,101.0,101-101,Dodoma,1.0,URBAN,Female_Dodoma _URBAN,Female,24.0,3.0,2217.420111,2067.27604
1,Female_Female_Dodoma _URBAN_101-101_10,101.0,101-101,Dodoma,1.0,URBAN,Female_Dodoma _URBAN,Female,21.0,2.0,2217.420111,2067.27604
2,Female_Female_Dodoma _URBAN_101-101_11,101.0,101-101,Dodoma,1.0,URBAN,Female_Dodoma _URBAN,Female,18.0,6.0,4434.840221,4134.55208
3,Female_Female_Dodoma _URBAN_101-101_12,101.0,101-101,Dodoma,1.0,URBAN,Female_Dodoma _URBAN,Female,16.0,6.0,2217.420111,2067.27604
4,Female_Female_Dodoma _URBAN_101-101_13,101.0,101-101,Dodoma,1.0,URBAN,Female_Dodoma _URBAN,Female,16.0,7.0,2217.420111,2067.27604
5,Female_Female_Dodoma _URBAN_101-101_14,101.0,101-101,Dodoma,1.0,URBAN,Female_Dodoma _URBAN,Female,18.0,6.0,2217.420111,2067.27604
6,Female_Female_Dodoma _URBAN_101-101_15,101.0,101-101,Dodoma,1.0,URBAN,Female_Dodoma _URBAN,Female,20.0,5.0,2217.420111,2067.27604
7,Female_Female_Dodoma _URBAN_101-101_16,101.0,101-101,Dodoma,1.0,URBAN,Female_Dodoma _URBAN,Female,18.0,5.0,2217.420111,2067.27604


,PUD_ID,PSU,CLUSTER,REGION,REGI_C,AREA,STRATA,SEX,AGE,NTOT,SAMPLEWEIGHT,HIVWEIGHT
6834,Female_Female_Simiyu _RURAL_403-403_23,403.0,403-403,Simiyu,24.0,RURAL,Female_Simiyu _RURAL,Female,13.0,7.0,4457.283339,NaN
1292,Female_Female_Dar es Salaam _URBAN_156-156_18,156.0,156-156,Dar es Salaam,7.0,URBAN,Female_Dar es Salaam _URBAN,Female,19.0,8.0,280.883708,285.662985
10141,Male_Male _Kaskazini Unguja_RURAL_879-879_22,879.0,879-879,Kaskazini Unguja,51.0,RURAL,Male _Kaskazini Unguja_RURAL,Male,15.0,3.0,134.105726,142.794406
8690,Male_Male _Tanga _RURAL_812-812_24,812.0,812-812,Tanga,4.0,RURAL,Male _Tanga _RURAL,Male,20.0,7.0,6788.838691,7714.441880
2954,Female_Female_Mbeya _URBAN_227-227_5,227.0,227-227,Mbeya,12.0,URBAN,Female_Mbeya _URBAN,Female,13.0,2.0,70.938354,NaN


### Household line, slot summaries, and duplicates

Derives **`HH_LINE`** / **`HH_KEY`**, prints per-slot summaries aligned with the harmonized slots, and lists duplicate **`PUD_ID`** rows.


In [16]:
import re

L = meta.column_names_to_labels or {}

df = df.copy()
df["HH_LINE"] = df["PUD_ID"].astype(str).str.split("_").str[-1]
df["HH_KEY"] = df["CLUSTER"].astype(str) + "_" + df["HH_LINE"]

# --- Per-variable summary format (compact) ---
# (1) Width: **character length** (min–max) for string/object columns — not “digit count” through the string.
#     For **numeric** columns only: digit count from integer string form (e.g. PSU codes) + min/max.
# (2) Optional **style** template when all values share one pattern (e.g. NNN-NNN for cluster IDs).
# (3) Male/Female text heuristic (not single-letter M/F codes) on string values.
# (4) dtype; n_distinct; missing; Stata label.

_WORD_SEX = re.compile(r"(?:male|females?|female)", re.IGNORECASE)
# Underscore-delimited segments like Female_… in long IDs (not single-letter TLQC/IQC codes "M", "F")
_EMBED_MF = re.compile(r"(?i)(?:^|_)(?:male|female)(?=_|$)")

def _abstract_digit_pattern(val: str) -> str:
    """Digit runs -> NNN; letter runs -> A; other chars literal (e.g. 133-133 -> NNN-NNN)."""
    parts = []
    i = 0
    while i < len(val):
        ch = val[i]
        if ch.isdigit():
            j = i
            while j < len(val) and val[j].isdigit():
                j += 1
            parts.append("N" * (j - i))
            i = j
        elif ch.isalpha():
            j = i
            while j < len(val) and val[j].isalpha():
                j += 1
            parts.append("A")
            i = j
        else:
            parts.append(ch)
            i += 1
    return "".join(parts)


def _unified_style_pattern(st: pd.Series):
    st = st.dropna().astype(str)
    if len(st) == 0:
        return None
    abstracts = st.map(_abstract_digit_pattern)
    if abstracts.nunique(dropna=False) != 1:
        return None
    pat = abstracts.iloc[0]
    if not any(ch.isdigit() for ch in pat):
        return None
    if set(pat) <= {"N"}:
        return None
    ex = st.iloc[0]
    if len(pat) > 72:
        return f"{pat[:72]}… (e.g. {ex[:40]}{'…' if len(ex) > 40 else ''})"
    return f"{pat} (e.g. {ex})"


def _width_note(s: pd.Series) -> str:
    sn = s.dropna()
    if len(sn) == 0:
        return "n/a"
    if pd.api.types.is_numeric_dtype(s):
        whole = (sn == sn.astype(float).astype(int)).all()
        if whole:
            lens = sn.astype(int).astype(str).str.len()
            lo, hi = int(lens.min()), int(lens.max())
            return f"{lo}-{hi} digits (integer codes)" if lo != hi else f"{lo} digits (integer codes)"
        lens = sn.astype(str).str.len()
        lo, hi = int(lens.min()), int(lens.max())
        return f"{lo}-{hi} chars (numeric as string)" if lo != hi else f"{lo} chars (numeric as string)"
    st = sn.astype(str)
    lens = st.str.len()
    lo, hi = int(lens.min()), int(lens.max())
    w = f"{lo}-{hi} chars" if lo != hi else f"{lo} chars"
    if st.str.fullmatch(r"\d+").all():
        return f"{w} (string; all numeric characters)"
    return f"{w} (string)"


def _special_id_note(s: pd.Series) -> str:
    if pd.api.types.is_numeric_dtype(s):
        return "no M/F identifier (numeric)"
    st = s.dropna().astype(str)
    if len(st) == 0:
        return "n/a"
    if st.str.contains(_WORD_SEX, regex=True, na=False).any() or st.str.contains(_EMBED_MF, regex=True, na=False).any():
        return "Male/Female text (words or _Female_/_Male_ segments)"
    return "no male/female text (heuristic)"


def slot_summary(title, cols, note_extra=""):
    """Print one slot: width (chars for text, digit-width note for numeric), optional style, dtype, counts."""
    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)
    miss = [c for c in cols if c not in df.columns]
    if miss:
        print("MISSING columns:", miss)
        return
    for c in cols:
        s = df[c]
        lbl = (L.get(c) or "")[:75]
        size_part = _width_note(s)
        style = _unified_style_pattern(s) if not pd.api.types.is_numeric_dtype(s) else None
        style_part = f"; style {style}" if style else ""
        id_part = _special_id_note(s)
        if pd.api.types.is_numeric_dtype(s):
            sn = s.dropna()
            extra = f"min/max={sn.min()}/{sn.max()}" if len(sn) else "min/max=n/a"
            info = f"{size_part}{style_part}; {id_part}; dtype={s.dtype}; {extra}"
        else:
            info = f"{size_part}{style_part}; {id_part}; dtype={s.dtype}"
        print(f"  {c} | {lbl}")
        print(f"    {info}; n_distinct={s.nunique(dropna=True)}; missing={s.isna().sum()}")
    if note_extra:
        print("  ", note_extra)


slot_summary("1. Respondent ID", ["PUD_ID"], "unique rows: " + str(df["PUD_ID"].nunique()) + f" / {len(df)}")
slot_summary("2. Household pieces", ["CLUSTER", "HH_LINE", "HH_KEY"], "dup HH_KEY: " + str(int(df["HH_KEY"].duplicated().sum())))
slot_summary("3. Geo level 1", ["REGION"])
slot_summary("4. Geo level 2 / urb-rural", ["AREA", "REGI_C"])
slot_summary("5. Cluster / PSU", ["CLUSTER", "PSU", "STRATA", "SAMPLEWEIGHT"])

print("\n--- Duplicate PUD_ID examples (if any) ---")
dups = df[df["PUD_ID"].duplicated(keep=False)].sort_values("PUD_ID")
if len(dups):
    _show = [c for c in ["PUD_ID", "CLUSTER", "HH_LINE", "REGION", "AREA", "SEX"] if c in dups.columns]
    display(dups[_show].head(25))
else:
    print("none")



1. Respondent ID
  PUD_ID | 
    44-47 chars (string); Male/Female text (words or _Female_/_Male_ segments); dtype=str; n_distinct=11353; missing=0
   unique rows: 11353 / 11414

2. Household pieces
  CLUSTER | 
    7 chars (string); no male/female text (heuristic); dtype=str; n_distinct=499; missing=0
  HH_LINE | 
    1-2 chars (string; all numeric characters); no male/female text (heuristic); dtype=object; n_distinct=25; missing=0
  HH_KEY | 
    9-10 chars (string); no male/female text (heuristic); dtype=str; n_distinct=11353; missing=0
   dup HH_KEY: 61

3. Geo level 1
  REGION | REGION
    4-16 chars (string); no male/female text (heuristic); dtype=str; n_distinct=31; missing=0

4. Geo level 2 / urb-rural
  AREA | Area
    5 chars (string); no male/female text (heuristic); dtype=str; n_distinct=2; missing=0
  REGI_C | 
    1-2 digits (integer codes); no M/F identifier (numeric); dtype=float64; min/max=1.0/55.0; n_distinct=31; missing=0

5. Cluster / PSU
  CLUSTER | 
    7 chars (

,PUD_ID,CLUSTER,HH_LINE,REGION,AREA,SEX
750,Female_Female_Dar es Salaam _URBAN_133-133_2,133-133,2,Dar es Salaam,URBAN,Female
757,Female_Female_Dar es Salaam _URBAN_133-133_2,133-133,2,Dar es Salaam,URBAN,Female
847,Female_Female_Dar es Salaam _URBAN_137-137_19,137-137,19,Dar es Salaam,URBAN,Female
860,Female_Female_Dar es Salaam _URBAN_137-137_19,137-137,19,Dar es Salaam,URBAN,Female
926,Female_Female_Dar es Salaam _URBAN_140-140_9,140-140,9,Dar es Salaam,URBAN,Female
927,Female_Female_Dar es Salaam _URBAN_140-140_9,140-140,9,Dar es Salaam,URBAN,Female
946,Female_Female_Dar es Salaam _URBAN_141-141_6,141-141,6,Dar es Salaam,URBAN,Female
947,Female_Female_Dar es Salaam _URBAN_141-141_6,141-141,6,Dar es Salaam,URBAN,Female
1226,Female_Female_Dar es Salaam _URBAN_153-153_4,153-153,4,Dar es Salaam,URBAN,Female
1227,Female_Female_Dar es Salaam _URBAN_153-153_4,153-153,4,Dar es Salaam,URBAN,Female


## 4. Harmonized codebook slots (Tanzania 2024)

File: `TANZANIA_VACS_2024_PUD.dta` under `data/raw/Tanzania 2024/`. For STRATA / CLUSTER / SAMPLEWEIGHT, see `TANZANIA_VACS_2024_DataUserGuide.pdf`.

Plain tab-separated block (select inside the fence, paste into Excel):

```
slot	pud_file	variables	type_and_width	notes
Respondent ID	TANZANIA_VACS_2024_PUD.dta	PUD_ID	str; 44-47 char length	11353 distinct / 11414 rows; 61 duplicate PUD_ID
Household ID	TANZANIA_VACS_2024_PUD.dta	CLUSTER + last underscore field of PUD_ID	str + str; CLUSTER 7 chars; segment 1-2 chars	Penultimate PUD_ID field equals CLUSTER all rows; 61 dup HH_KEY; no hh column
Geo level 1	TANZANIA_VACS_2024_PUD.dta	REGION	str; 4-16 char length	31 regions
Geo level 2	TANZANIA_VACS_2024_PUD.dta	AREA (optional REGI_C)	AREA str 5 chars (URBAN/RURAL); REGI_C float 1-31	
Cluster	TANZANIA_VACS_2024_PUD.dta	CLUSTER; PSU	CLUSTER str; 7 chars; style NNN-NNN (e.g. 133-133); PSU numeric (101–931)	svy: cluster=CLUSTER, strata=STRATA, weight=SAMPLEWEIGHT
Interview date	(not in PUD)	—	—	Fieldwork Mar-Jun 2024 per guide; no person-level date in dta
```

Excel note (Household ID): Type str + str; CLUSTER is 7 characters; line within cluster is 1–2 characters (last underscore-delimited field of PUD_ID; often numeric digits only).

Notebook: `HH_LINE` = last PUD_ID segment; `HH_KEY` = CLUSTER + "_" + HH_LINE.
